# Data merge and feature engineering


In [11]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Uw11\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [13]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")

In [69]:
df_weather.shape

(844417, 34)

In [33]:
df_war_events.tail()

,datetime_hour,region_id,region_key,alarm_minutes_in_hour,alarm_active
924659,2026-03-16 23:00:00,22,Хмельницька,0.000000,0
924660,2026-03-16 23:00:00,23,Черкаська,0.000000,0
924661,2026-03-16 23:00:00,24,Чернівецька,0.000000,0
924662,2026-03-16 23:00:00,25,Чернігівська,60.000000,1
924663,2026-03-16 23:00:00,26,Київ,17.216667,1


In [34]:
df_isw_matrix.tail()

,day_datetime,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
1450,2026-02-25,0.794526,0.278399,0.076717,-0.017046,0.130514,0.027260,-0.016811,0.133450,-0.089720,0.070505,-0.001508,0.029849,0.216945,-0.191913,0.039367,-0.081793,-0.028576,-0.015380,-0.037350,0.051588,0.033024,-0.001077,-0.032285,0.047213,0.026472,-0.034078,0.021475,-0.026009,0.012121,-0.001979,-0.006542,0.006595,0.003465,-0.026377,-0.018966,-0.012102,0.025430,0.011179,-0.000786,-0.006697,-0.002752,-0.014308,-0.027570,0.049736,-0.027920,-0.019518,-0.013256,-0.027406,0.001268,-0.024612,-0.021970,0.019933,0.004401,-0.070484,0.019840,-0.026222,-0.025895,-0.010331,-0.010895,0.025983,0.005557,-0.012166,-0.001707,0.026743,0.005436,-0.004494,0.005406,0.014124,-0.012217,-0.003826,0.004201,0.022561,-0.012841,0.007031,0.028518,0.016360,0.005313,-0.008868,-0.030852,-0.008994,0.020207,0.019844,0.009032,-0.003323,0.009689,0.006533,0.014164,-0.001182,0.023723,-0.045677,-0.010081,-0.001688,-0.036899,-0.017436,-0.003877,-0.015655,-0.010086,0.007955,0.020829,-0.018619,-0.003883,0.006805,-0.016228,0.003644,-0.008055,-0.002014,0.009305,0.016573,-0.006390,0.006504,-0.016563,-0.016203,-0.009431,-0.017402,0.011307,0.019392,0.036467,-0.002640,-0.006597,-0.020570,-0.027470,0.014320,0.009700,-0.004638,0.004841,0.004362,-0.000251,0.006573,-0.000536,0.006895,-0.002910,0.014939,-0.008813,0.014757,0.010208,0.016375,-0.001539,-0.000318,0.012399,-0.004559,-0.003080,-0.031158,-0.023643,0.000771,0.023196,0.010538,-0.017646,-0.024978,0.006705,0.014877
1451,2026-02-26,0.773297,0.254389,0.092057,-0.014604,0.112812,0.023849,-0.033479,0.117877,-0.090861,0.101495,-0.002202,0.049757,0.236446,-0.212107,0.065014,-0.087469,-0.021346,-0.007087,-0.051019,0.065923,-0.026802,-0.081054,0.017684,0.068441,0.035285,-0.012822,-0.033085,0.032736,0.058368,-0.021791,0.007965,-0.023264,-0.051200,-0.022218,-0.018411,0.018479,0.016588,0.042810,0.033631,-0.003018,0.031416,0.002899,0.028572,0.024571,-0.009660,-0.011085,0.016730,-0.023477,0.014187,0.014596,0.007376,0.003512,-0.019419,-0.039254,0.000321,-0.008324,-0.00

In [14]:
pd.set_option('display.max_columns', None)

In [15]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [16]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour
651221,2025-04-01 00:00:00,7,4.8,0.006,100.0,4.17,0.10,9.6,8.7,70.59,4.5,0.0,0.0,0.0,0.0,25.6,7.3,10.0,1015.1,89.9,0.0,0.0,0.0,False,True,False,False,1,0,Uzhgorod,2025-04-01,07:08:33,20:01:49,Закарпатська,0,0.000000
315675,2023-08-26 03:00:00,5,11.6,0.000,0.0,0.00,0.34,13.5,13.5,90.05,11.9,0.0,0.0,0.0,0.0,10.4,4.7,34.4,1017.0,23.9,0.0,0.0,0.0,False,True,False,False,5,3,Donetsk,2023-08-26,05:37:50,19:22:39,Донецька,0,0.000000
726023,2025-08-14 12:00:00,14,9.9,0.000,0.0,0.00,0.68,29.7,29.2,38.80,14.2,0.0,0.0,0.0,0.0,23.0,14.4,50.0,1021.3,11.5,766.8,2.8,8.0,True,False,False,False,3,12,Mykolaiv,2025-08-14,05:48:06,20:04:35,Миколаївська,0,0.000000
7947,2022-03-09 19:00:00,5,-6.6,0.600,100.0,20.83,0.21,-1.0,-3.8,66.76,-6.4,0.0,0.0,0.1,1.7,15.1,7.6,33.4,1015.0,93.7,126.0,0.5,0.0,False,True,False,False,2,19,Donetsk,2022-03-09,05:54:28,17:24:58,Донецька,0,0.000000
423518,2024-02-29 08:00:00,17,4.6,0.000,0.0,0.00,0.66,5.8,5.8,85.90,3.7,0.0,0.0,0.0,0.0,13.7,3.6,181.0,1023.0,99.8,31.1,0.1,0.0,False,True,False,False,3,8,Rivne,2024-02-29,07:00:26,17:55:22,Рівненська,0,0.000000
74808,2022-07-03 22:00:00,2,13.2,2.500,100.0,20.83,0.13,22.0,22.0,67.89,15.8,0.0,0.0,0.0,0.0,9.4,4.7,67.9,1018.0,50.9,0.0,0.1,0.0,False,True,False,False,6,22,Vinnytsia,2022-07-03,05:06:17,21:14:20,Вінницька,0,0.000000
372197,2023-12-02 06:00:00,7,0.4,22.162,100.0,91.67,0.66,1.2,-2.5,92.66,0.2,0.0,0.0,0.0,3.8,14.8,12.9,360.0,1000.8,100.0,0.0,0.0,0.0,False,True,False,False,5,6,Uzhgorod,2023-12-02,08:02:50,16:37:14,Закарпатська,0,0.000000
18094,2022-03-27 10:00:00,25,-8.7,0.100,100.0,4.17,0.82,-0.3,-7.3,56.52,-7.9,0.0,0.0,0.0,0.0,64.4,33.5,332.0,1018.0,100.0,279.0,1.0,3.0,False,True,False,False,6,10,Chernihiv,2022-03-27,06:41:57,19:19:32,Чернігівська,0,0.000000
151586,2022-11-14 05:00:00,4,2.0,0.000,0.0,0.00,0.66,5.2,3.6,88.15,3.4,0.0,0.0,0.0,0.0,18.4,6.8,332.8,1025.0,9.4,0.0,0.1,0.0,True,False,False,False,0,5,Dnipro,2022-11-14,06:45:55,16:02:14,Дніпропетровська,0,0.000000
808069,2026-01-10 03:00:00,19,-12.9,0.000,0.0,0.00,0.75,-9.1,-15.2,85.39,-11.1,0.0,0.0,0.0,35.2,27.0,13.3,278.6,1001.0,81.9,0.0,0.0,0.0,False,True,False,False,5,3,Ternopil,2026-01-10,08:11:48,16:38:56,Тернопільська,0,0.000000


In [17]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,-0.3,0.3,100.0,4.17,0.77,2.1,-0.5,91.76,0.9,0.0,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,True,False,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,-0.3,0.3,100.0,4.17,0.77,2.1,-0.4,89.80,0.6,0.0,0.0,0.0,0.0,16.6,8.3,289.2,1021.0,97.9,0.0,0.1,0.0,False,True,False,False,3,1,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,-0.3,0.3,100.0,4.17,0.77,2.0,-0.3,90.44,0.6,0.0,0.0,0.0,0.0,27.7,7.6,287.6,1021.0,98.2,0.0,0.1,0.0,False,True,False,False,3,2,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,-0.3,0.3,100.0,4.17,0.77,1.9,-0.3,91.75,0.7,0.0,0.0,0.0,0.0,15.1,7.2,288.9,1021.0,98.8,0.0,0.1,0.0,False,True,False,False,3,3,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,-0.3,0.3,100.0,4.17,0.77,1.8,-0.2,92.40,0.7,0.0,0.0,0.0,0.0,13.7,6.5,290.4,1021.0,100.0,0.0,0.1,0.0,False,True,False,False,3,4,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0


In [18]:
df_final.tail()

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour
844324,2026-03-16 19:00:00,26,-1.8,0.0,0.0,0.0,0.92,7.5,7.5,55.25,-0.9,0.0,0.0,0.0,0.0,10.4,3.6,80.0,1019.0,44.3,5.5,0.1,0.0,False,True,False,False,0,19,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000
844347,2026-03-16 20:00:00,26,-1.8,0.0,0.0,0.0,0.92,6.9,6.9,59.27,-0.5,0.0,0.0,0.0,0.0,9.4,4.0,79.6,1021.0,59.3,5.5,0.1,0.0,False,True,False,False,0,20,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000
844370,2026-03-16 21:00:00,26,-1.8,0.0,0.0,0.0,0.92,6.1,6.1,62.62,-0.5,0.0,0.0,0.0,0.0,10.1,3.6,88.3,1021.0,39.4,5.5,0.1,0.0,False,True,False,False,0,21,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000
844393,2026-03-16 22:00:00,26,-1.8,0.0,0.0,0.0,0.92,5.3,5.3,65.23,-0.7,0.0,0.0,0.0,0.0,9.0,2.5,85.6,1021.0,19.4,5.5,0.1,0.0,True,False,False,False,0,22,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,1,25.650000
844416,2026-03-16 23:00:00,26,-1.8,0.0,0.0,0.0,0.92,4.7,4.7,66.06,-1.1,0.0,0.0,0.0,0.0,5.8,2.2,63.2,1021.0,10.6,5.5,0.1,0.0,True,False,False,False,0,23,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,1,17.216667


In [19]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 844417 entries, 0 to 844416
Data columns (total 36 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  844417 non-null  datetime64[us]
 1   region_id                      844417 non-null  int64         
 2   day_dew                        844417 non-null  float64       
 3   day_precip                     844417 non-null  float64       
 4   day_precipprob                 844417 non-null  float64       
 5   day_precipcover                844417 non-null  float64       
 6   day_moonphase                  844417 non-null  float64       
 7   hour_temp                      844417 non-null  float64       
 8   hour_feelslike                 844417 non-null  float64       
 9   hour_humidity                  844417 non-null  float64       
 10  hour_dew                       844417 non-null  float64       
 11  hour_precip     

In [20]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)
#df_final['alarm_minutes_lag1'] = (df_final.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))
#df_final['alarm_minutes_lag3'] = (df_final.groupby('region_id')['alarm_minutes_in_hour'].shift(3).fillna(0))

lag_cols = ["alarm_lag_1","alarm_lag_3","alarm_lag_6","alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

df_final['alarms_in_last_24h'] = (df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum()))

df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)

df_final.sample(10)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1
671434,2025-05-07 15:00:00,2,6.9,4.000,100.0,4.17,0.32,12.2,12.2,70.05,6.9,0.000,0.0,0.0,0.0,29.9,7.2,360.0,1013.6,87.5,184.2,0.7,2.0,False,True,False,False,2,15,Vinnytsia,2025-05-07,05:34:47,20:31:47,Вінницька,0,0.000000,0.0,1.0,1.0,0.0,8.0,0,0,2.0
552886,2024-10-10 23:00:00,25,11.1,0.000,0.0,0.00,0.25,13.8,13.8,85.44,11.4,0.000,0.0,0.0,0.0,25.2,14.4,140.7,1008.0,81.0,0.0,0.0,0.0,False,True,False,False,3,23,Chernihiv,2024-10-10,07:10:52,18:11:46,Чернігівська,0,0.000000,0.0,1.0,0.0,1.0,13.0,0,1,12.0
629071,2025-02-20 06:00:00,9,-8.7,0.100,100.0,4.17,0.75,-5.0,-9.1,86.52,-6.9,0.000,0.0,0.1,0.4,17.3,9.4,312.5,1035.0,92.0,0.0,0.0,0.0,False,True,False,False,3,6,Ivano-Frankivsk,2025-02-20,07:20:18,17:50:15,Івано-Франківська,1,19.583333,0.0,0.0,0.0,0.0,0.0,0,1,14.0
348413,2023-10-21 23:00:00,7,12.4,0.093,100.0,4.17,0.23,15.3,15.3,88.10,13.4,0.093,100.0,0.0,0.0,20.2,4.8,69.0,1008.2,82.3,0.0,0.0,0.0,False,False,True,False,5,23,Uzhgorod,2023-10-21,07:59:22,18:30:59,Закарпатська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,1,4.0
366569,2023-11-22 11:00:00,20,-11.2,0.300,100.0,4.17,0.32,-3.9,-7.1,62.66,-10.0,0.000,0.0,0.0,8.2,11.2,7.3,271.0,1020.8,90.0,274.6,1.0,3.0,False,True,False,False,2,11,Kharkiv,2023-11-22,06:58:16,15:43:29,Харківська,1,5.983333,0.0,1.0,0.0,1.0,12.0,0,0,0.0
73044,2022-06-30 20:00:00,15,18.1,17.000,100.0,4.17,0.03,25.5,25.5,70.31,19.7,0.000,0.0,0.0,0.0,14.4,5.4,108.1,1014.0,100.0,128.0,0.5,1.0,False,True,False,False,3,20,Odesa,2022-06-30,05:07:39,20:53:38,Одеська,0,0.000000,0.0,0.0,1.0,0.0,5.0,0,0,0.0
570757,2024-11-11 00:00:00,16,0.8,0.100,100.0,4.17,0.34,2.1,-0.6,87.87,0.3,0.000,0.0,0.0,0.0,16.2,9.0,34.5,1032.0,99.3,0.0,0.0,0.0,False,True,False,False,0,0,Poltava,2024-11-11,06:47:18,16:03:52,Полтавська,1,60.000000,1.0,1.0,0.0,0.0,14.0,0,1,7.0
70822,2022-06-26 23:00:00,25,13.5,0.000,0.0,0.00,0.89,21.7,21.7,64.01,14.6,0.000,0.0,0.0,0.0,12.6,6.8,57.5,1020.0,100.0,0.0,0.1,0.0,False,True,False,False,6,23,Chernihiv,2022-06-26,04:39:10,21:16:05,Чернігівська,0,0.000000,0.0,0.0,0.0,1.0,10.0,1,1,1.0
630722,2025-02-23 03:00:00,4,-11.9,0.000,0.0,0.00,0.85,-9.2,-15.6,78.81,-12.2,0.000,0.0,0.0,3.1,23.4,14.0,337.4,1035.0,13.6,0.0,0.0,0.0,True,False,False,False,6,3,Dnipro,2025-02-23,06:32:52,17:14:32,Дніпропетровська,1,36.466667,1.0,1.0,1.0,0.0,11.0,1,1,15.0
81565,2022-07-15 15:00:00,16,15.2,0.700,100.0,4.17,0.53,28.2,27.9,41.51,13.9,0.000,0.0,0.0,0.0,29.2,10.8,210.0,1011.5,80.0,733.0,2.6,7.0,False,True,False,False,4,15,Poltava,2022-07-15,04:51:01,20:43:57,Полтавська,0,0.000000,0.0,0.0,0.0,1.0,8.0,0,0,1.0


In [21]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(
    index='datetime_hour',
    columns='region_id',
    values='alarm_active',
    fill_value=0
)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)

neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())

neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']

df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')

In [22]:
def hours_since_last_alarm(series):
    shifted = series.shift(1).fillna(0)
    result = []
    count = 0
    for val in shifted:
        if val == 1:
            count = 0
        else:
            count += 1
        result.append(count)
    return pd.Series(result, index=series.index)

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(hours_since_last_alarm))

In [23]:
df_final.sample(20)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
235100,2025-08-31 23:00:00,8,14.8,0.000,0.0,0.00,0.25,24.7,24.7,65.90,17.9,0.00,0.0,0.0,0.0,24.8,15.5,176.8,1007.0,0.0,0.0,0.0,0.0,True,False,False,False,6,23,Zaporozhye,2025-08-31,05:56:15,19:22:13,Запорізька,1,60.000000,1.0,0.0,0.0,1.0,16.0,1,1,5.0,1.0,0
781620,2023-02-06 04:00:00,25,-6.5,0.100,100.0,4.17,0.52,-3.0,-8.3,82.87,-5.5,0.00,0.0,0.0,2.6,24.5,15.5,356.7,1032.0,97.0,0.0,0.0,0.0,False,True,False,False,0,4,Chernihiv,2023-02-06,07:25:43,16:52:48,Чернігівська,0,0.000000,0.0,0.0,0.0,0.0,2.0,0,1,0.0,0.0,18
7214,2022-12-21 15:00:00,2,-3.7,0.000,0.0,0.00,0.91,2.2,0.2,73.75,-2.0,0.00,0.0,0.0,0.1,14.4,6.8,175.5,1019.0,100.0,112.0,0.4,1.0,False,True,False,False,2,15,Vinnytsia,2022-12-21,07:58:38,16:09:54,Вінницька,1,8.183333,1.0,1.0,0.0,0.0,3.0,0,0,24.0,7.0,0
584594,2024-12-18 20:00:00,19,3.7,3.000,100.0,4.17,0.60,5.7,5.7,86.97,3.7,0.00,0.0,0.0,0.0,19.8,0.0,230.6,1019.4,80.0,0.0,0.0,0.0,False,True,False,False,2,20,Ternopil,2024-12-18,08:10:05,16:18:53,Тернопільська,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,3.0,0.0,106
348979,2022-06-06 11:00:00,13,11.1,0.000,0.0,0.00,0.22,20.5,20.5,55.17,11.2,0.00,0.0,0.0,0.0,28.1,11.5,132.7,1020.0,61.3,697.0,2.5,7.0,False,True,False,False,0,11,Lviv,2022-06-06,05:17:38,21:28:03,Львівська,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,8.0,0.0,27
362967,2024-01-10 08:00:00,13,-15.6,0.000,0.0,0.00,0.97,-17.1,-17.1,85.14,-19.0,0.00,0.0,0.0,6.0,15.8,0.0,279.1,1036.7,8.5,0.0,0.0,0.0,True,False,False,False,2,8,Lviv,2024-01-10,08:19:31,16:43:16,Львівська,1,50.216667,0.0,0.0,0.0,0.0,0.0,0,0,13.0,3.0,47
399736,2024-02-29 13:00:00,14,-0.5,0.000,0.0,0.00,0.66,9.5,9.5,56.18,1.2,0.00,0.0,0.0,0.0,16.6,4.0,132.9,1026.0,90.6,547.5,2.0,5.0,False,True,False,False,3,13,Mykolaiv,2024-02-29,06:33:16,17:36:35,Миколаївська,1,10.866667,0.0,0.0,0.0,0.0,4.0,0,0,4.0,1.0,13
610915,2023-11-29 16:00:00,20,0.2,17.000,100.0,8.33,0.57,1.0,-5.8,100.00,1.0,0.00,0.0,0.2,4.2,54.0,36.0,130.0,996.0,100.0,1.2,0.0,0.0,False,True,False,False,2,16,Kharkiv,2023-11-29,07:08:35,15:37:36,Харківська,0,0.000000,0.0,0.0,0.0,0.0,7.0,0,0,0.0,0.0,12
585510,2025-01-26 00:00:00,19,2.3,0.000,0.0,0.00,0.90,2.4,-0.7,100.00,2.4,0.00,0.0,0.0,0.0,21.6,10.8,154.1,1021.0,100.0,0.0,0.0,0.0,False,True,False,False,6,0,Ternopil,2025-01-26,07:57:39,17:03:28,Тернопільська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,1,12.0,0.0,41
561614,2022-05-06 06:00:00,19,-0.2,0.000,0.0,0.00,0.17,6.1,6.1,63.08,-0.4,0.00,0.0,0.0,0.0,13.3,0.0,138.4,1025.1,14.3,0.0,0.1,0.0,True,False,False,False,4,6,Ternopil,2022-05-06,05:47:13,20:42:22,Тернопільська,0,0.000000,0.0,0.0,0.0,0.0,4.0,0,1,4.0,0.0,8


In [24]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
575,2023-09-24,0.790050,-0.136141,0.019131,-0.165198,-0.047305,0.007962,-0.063581,0.080242,0.009071,-0.046490,0.039469,0.033295,0.155179,0.223246,0.068429,-0.085887,0.021341,0.041823,0.065208,0.003649,0.038336,0.027303,-0.109591,-0.014070,-0.016398,0.010042,-0.015257,-0.007831,-0.027794,0.013924,-0.010308,-0.006039,0.009242,0.015530,-0.038670,0.013095,-0.026556,0.017733,-0.008279,0.029124,-0.007625,0.030991,-0.011718,0.030206,-0.004725,-0.043405,-0.013866,-0.036863,-0.004565,0.017346,0.024288,0.001964,-0.017125,0.018723,0.009956,-0.008714,-0.006790,-0.053065,0.006683,-0.029119,0.004267,-0.005652,0.000528,0.020887,0.013472,0.051162,0.020877,-0.011886,0.025595,-0.061849,0.026148,0.016998,-0.035713,0.014083,-0.024563,0.037633,-0.020835,0.005519,-0.023980,-0.035664,0.015878,-0.014599,0.032550,0.022425,0.033564,0.011083,0.020455,-0.007790,-0.000020,-0.009774,-0.036911,-0.047396,-0.032949,-0.009683,-0.042611,0.002180,0.015359,-0.027570,0.005814,0.020127,-0.004326,-0.000988,0.007083,0.042176,-0.022176,0.007157,0.041651,0.018320,0.038056,0.032019,-0.021284,0.032283,-0.033734,-0.016115,0.036548,0.002318,-0.032854,0.012726,-0.029057,-0.004690,-0.002499,0.010582,-0.000463,0.001856,0.016354,0.009675,0.010523,-0.015037,0.003037,0.001309,-0.008433,0.003542,0.024139,0.005638,-0.000382,-0.005976,-0.000292,-0.017312,-0.024015,-0.001086,0.035635,-0.002946,-0.014342,0.003722,0.008628,-0.011754,0.027997,-0.019280,-0.018877,-0.011170
56,2022-04-20,0.620680,-0.172529,0.345004,0.226888,0.051193,-0.096838,-0.016490,-0.112101,-0.081036,-0.082493,0.196736,-0.049930,0.000839,0.007660,0.055525,-0.082571,0.096303,-0.062868,0.028160,-0.070337,-0.002592,0.040318,0.093280,0.050480,-0.031456,-0.033413,-0.077659,-0.067716,-0.016606,-0.017769,-0.006578,-0.036763,-0.035211,0.005416,-0.006933,-0.014326,-0.007392,0.007991,-0.021543,0.014500,0.033333,-0.037115,-0.000332,0.044117,0.015302,0.024073,0.009926,0.047272,-0.000352,-0.016995,-0.041864,0.018904,0.029753,-0.029651,-0.019270,0.060047,0.020191,-0.021356

In [30]:
df_isw_matrix.tail()

,day_datetime,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
1450,2026-02-25,0.794526,0.278399,0.076717,-0.017046,0.130514,0.027260,-0.016811,0.133450,-0.089720,0.070505,-0.001508,0.029849,0.216945,-0.191913,0.039367,-0.081793,-0.028576,-0.015380,-0.037350,0.051588,0.033024,-0.001077,-0.032285,0.047213,0.026472,-0.034078,0.021475,-0.026009,0.012121,-0.001979,-0.006542,0.006595,0.003465,-0.026377,-0.018966,-0.012102,0.025430,0.011179,-0.000786,-0.006697,-0.002752,-0.014308,-0.027570,0.049736,-0.027920,-0.019518,-0.013256,-0.027406,0.001268,-0.024612,-0.021970,0.019933,0.004401,-0.070484,0.019840,-0.026222,-0.025895,-0.010331,-0.010895,0.025983,0.005557,-0.012166,-0.001707,0.026743,0.005436,-0.004494,0.005406,0.014124,-0.012217,-0.003826,0.004201,0.022561,-0.012841,0.007031,0.028518,0.016360,0.005313,-0.008868,-0.030852,-0.008994,0.020207,0.019844,0.009032,-0.003323,0.009689,0.006533,0.014164,-0.001182,0.023723,-0.045677,-0.010081,-0.001688,-0.036899,-0.017436,-0.003877,-0.015655,-0.010086,0.007955,0.020829,-0.018619,-0.003883,0.006805,-0.016228,0.003644,-0.008055,-0.002014,0.009305,0.016573,-0.006390,0.006504,-0.016563,-0.016203,-0.009431,-0.017402,0.011307,0.019392,0.036467,-0.002640,-0.006597,-0.020570,-0.027470,0.014320,0.009700,-0.004638,0.004841,0.004362,-0.000251,0.006573,-0.000536,0.006895,-0.002910,0.014939,-0.008813,0.014757,0.010208,0.016375,-0.001539,-0.000318,0.012399,-0.004559,-0.003080,-0.031158,-0.023643,0.000771,0.023196,0.010538,-0.017646,-0.024978,0.006705,0.014877
1451,2026-02-26,0.773297,0.254389,0.092057,-0.014604,0.112812,0.023849,-0.033479,0.117877,-0.090861,0.101495,-0.002202,0.049757,0.236446,-0.212107,0.065014,-0.087469,-0.021346,-0.007087,-0.051019,0.065923,-0.026802,-0.081054,0.017684,0.068441,0.035285,-0.012822,-0.033085,0.032736,0.058368,-0.021791,0.007965,-0.023264,-0.051200,-0.022218,-0.018411,0.018479,0.016588,0.042810,0.033631,-0.003018,0.031416,0.002899,0.028572,0.024571,-0.009660,-0.011085,0.016730,-0.023477,0.014187,0.014596,0.007376,0.003512,-0.019419,-0.039254,0.000321,-0.008324,-0.00

In [25]:
#df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)    ТУТ ЗСУВ ІСВ НА + 1 ДЕНЬ
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [26]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [27]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [28]:
df_final.head(10)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,-0.3,0.3,100.0,4.17,0.77,2.1,-0.5,91.76,0.9,0.0,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,True,False,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0,0.0,0.0,0.0,0.0,NaN,0,1,NaN,NaN,1,0.648336,-0.079572,0.178041,0.061854,0.073453,-0.055809,-0.011311,0.047310,0.008354,0.006474,-0.047099,0.020496,0.050190,-0.038573,0.057228,-0.072471,0.045393,-0.116783,0.168249,0.167355,-0.025072,-0.073858,-0.097024,-0.146191,0.027983,0.082989,0.014423,0.058370,-0.033928,0.046265,0.006901,-0.049498,0.126117,-0.089039,-0.006767,-0.031686,-0.010346,0.025637,0.061278,0.162580,-0.028081,0.161291,-0.018217,-0.002657,0.038485,-0.000530,0.044413,0.107176,-0.050721,0.064279,-0.066677,0.016742,-0.050257,0.072549,-0.035201,-0.027279,0.017115,0.043631,0.024660,-0.025841,0.031320,-0.076959,-0.072986,-0.040465,0.003958,-0.022475,0.033208,-0.028056,-0.005499,-0.055998,-0.021882,-0.070462,0.018237,-0.031396,-0.074606,0.072951,0.039997,0.036423,-0.027152,-0.071900,0.026267,-0.046270,-0.017945,0.007933,-0.006203,-0.046099,0.032793,-0.069113,0.020403,0.030783,-0.003787,-0.000490,-0.022951,0.013498,0.009664,-0.007012,-0.016325,-0.005732,0.036069,-0.063906,0.033891,0.010560,-0.104517,-0.022069,0.054322,-0.005460,-0.029093,0.040991,-0.053018,-0.024017,0.050817,-0.003297,-0.020895,-0.05

In [29]:
df_final.isna().sum()

datetime_hour         0
region_id             0
day_dew               0
day_precip            0
day_precipprob        0
                  ...  
isw_topic_145     15696
isw_topic_146     15696
isw_topic_147     15696
isw_topic_148     15696
isw_topic_149     15696
Length: 196, dtype: int64

In [31]:
df_final.tail()

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
844988,2026-03-16 19:00:00,26,-1.8,0.0,0.0,0.0,0.92,7.5,7.5,55.25,-0.9,0.0,0.0,0.0,0.0,10.4,3.6,80.0,1019.0,44.3,5.5,0.1,0.0,False,True,False,False,0,19,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000,0.0,0.0,0.0,0.0,3.0,0,0,8.0,1.0,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
844989,2026-03-16 20:00:00,26,-1.8,0.0,0.0,0.0,0.92,6.9,6.9,59.27,-0.5,0.0,0.0,0.0,0.0,9.4,4.0,79.6,1021.0,59.3,5.5,0.1,0.0,False,True,False,False,0,20,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000,0.0,0.0,0.0,1.0,3.0,0,0,11.0,1.0,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [35]:
df_final.fillna(0, inplace=True)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,-0.3,0.3,100.0,4.17,0.77,2.1,-0.5,91.76,0.9,0.0,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,True,False,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,1,0.648336,-0.079572,0.178041,0.061854,0.073453,-0.055809,-0.011311,0.047310,0.008354,0.006474,-0.047099,0.020496,0.050190,-0.038573,0.057228,-0.072471,0.045393,-0.116783,0.168249,0.167355,-0.025072,-0.073858,-0.097024,-0.146191,0.027983,0.082989,0.014423,0.058370,-0.033928,0.046265,0.006901,-0.049498,0.126117,-0.089039,-0.006767,-0.031686,-0.010346,0.025637,0.061278,0.162580,-0.028081,0.161291,-0.018217,-0.002657,0.038485,-0.000530,0.044413,0.107176,-0.050721,0.064279,-0.066677,0.016742,-0.050257,0.072549,-0.035201,-0.027279,0.017115,0.043631,0.024660,-0.025841,0.031320,-0.076959,-0.072986,-0.040465,0.003958,-0.022475,0.033208,-0.028056,-0.005499,-0.055998,-0.021882,-0.070462,0.018237,-0.031396,-0.074606,0.072951,0.039997,0.036423,-0.027152,-0.071900,0.026267,-0.046270,-0.017945,0.007933,-0.006203,-0.046099,0.032793,-0.069113,0.020403,0.030783,-0.003787,-0.000490,-0.022951,0.013498,0.009664,-0.007012,-0.016325,-0.005732,0.036069,-0.063906,0.033891,0.010560,-0.104517,-0.022069,0.054322,-0.005460,-0.029093,0.040991,-0.053018,-0.024017,0.050817,-0.003297,-0.020895,

In [36]:
df_final.isna().sum()

datetime_hour     0
region_id         0
day_dew           0
day_precip        0
day_precipprob    0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 196, dtype: int64

### Feature engineering for isw topics 

In [37]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = (
    df_final.groupby('region_id')['isw_total_intensity']
    .diff(24)
    .fillna(0)
)

df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\1269306822.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\1269306822.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\1269306822.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [38]:
df_final.sample(10)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
717349,2023-10-28 22:00:00,23,8.5,2.600,100.0,20.83,0.50,12.0,12.0,69.05,6.5,0.0,0.0,0.0,0.0,38.2,18.7,275.7,1006.0,4.2,0.0,0.0,0.0,True,False,False,False,5,22,Cherkasy,2023-10-28,07:32:48,17:37:36,Черкаська,0,0.00,0.0,0.0,0.0,1.0,4.0,1,0,6.0,0.0,9,0.780125,-0.078214,-0.107154,-0.200354,-0.047502,-0.162899,-0.198176,-0.062243,-0.143992,-0.057412,-0.090597,-0.033255,-0.106709,-0.041674,0.032578,-0.130523,0.050504,0.053701,0.001499,-0.061491,-0.056953,0.008248,-0.054784,0.006040,0.045104,0.053447,-0.034620,-0.064085,0.038855,-0.047638,-0.001892,-0.048556,0.002408,-0.015156,0.016534,-0.004498,0.032239,-0.036042,0.025961,0.001280,0.059896,0.008775,0.026882,0.008909,0.034830,0.002943,-0.050798,0.011676,0.033403,0.043676,0.007134,-0.029772,0.018775,-0.016045,-0.005306,0.018814,0.032342,0.004165,0.027896,-0.035272,0.004270,-0.032764,0.018601,0.001412,0.006088,0.037498,-0.037997,-0.011988,0.015662,-0.004225,0.010505,-0.005078,0.001676,0.024582,0.044064,0.019680,-0.000681,0.028726,0.023436,0.002041,0.003866,-0.021816,0.014209,-0.013614,0.021637,-0.023041,-0.008005,0.004920,0.002885,-0.026602,0.007864,0.029133,-0.000101,0.002658,0.021538,0.027345,-0.003936,-0.006009,-0.008835,0.005523,0.002450,0.

### TELEGRAM MERGE

In [39]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [40]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.006988,0.000199,0.019066,0.000954,-0.008573,0.027555,-0.023391,0.082142,0.048557,-0.035440,-0.044857,0.023973,-0.010295,0.314186,0.441313,0.148148,-0.033453,0.013616,-0.015178,0.300137,-0.013802,0.024118,0.032728,-0.075552,-0.055436,0.056107,-0.114964,-0.062816,0.000904,0.040613,0.030641,-0.011243,-0.031934,-0.016560,-0.000001,-0.009350,-0.013027,0.023568,-0.014856,-0.013250,0.003955,0.006923,0.018694,-0.040851,-0.016602,-0.022611,0.020255,-0.044764,-0.014433,-0.000230,-0.005441,-0.015143,0.000333,0.033882,-0.001488,0.004348,0.028966,0.024959,-0.010554,-0.007216,-0.016782,-0.017824,-0.004213,-0.016672,-0.025594,0.012135,0.018105,-0.023220,0.009999,0.012852,-0.056985,-0.036515,-0.008792,-0.007234,-0.026848,-0.062293,0.085060,0.036042,-0.086716,-0.011956,-0.004898,0.141366,-0.055466,-0.088688,0.061815

In [41]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\2851151304.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")


In [42]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]

tg_hourly = (
    df_tg_matrix.groupby("datetime_hour")[topic_cols]
    .mean()
    .sort_index()
    .reset_index()
)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\2188712141.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


In [43]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.006988,0.000199,0.019066,0.000954,-0.008573,0.027555,-0.023391,0.082142,0.048557,-0.035440,-0.044857,0.023973,-0.010295,0.314186,0.441313,0.148148,-0.033453,0.013616,-0.015178,0.300137,-0.013802,0.024118,0.032728,-0.075552,-0.055436,0.056107,-0.114964,-0.062816,0.000904,0.040613,0.030641,-0.011243,-0.031934,-0.016560,-0.000001,-0.009350,-0.013027,0.023568,-0.014856,-0.013250,0.003955,0.006923,0.018694,-0.040851,-0.016602,-0.022611,0.020255,-0.044764,-0.014433,-0.000230,-0.005441,-0.015143,0.000333,0.033882,-0.001488,0.004348,0.028966,0.024959,-0.010554,-0.007216,-0.016782,-0.017824,-0.004213,-0.016672,-0.025594,0.012135,0.018105,-0.023220,0.009999,0.012852,-0.056985,-0.036515,-0.008792,-0.007234,-0.026848,-0.062293,0.085060,0.036042,-0.086716,-0.011956,-0.004898,0.141366,-0.055466,-0.0

In [44]:
#tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)        ТУТ ЗСУВ ТГ НА 1 ГОДИНУ 

In [45]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [46]:
df_final.sample(20)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg

In [47]:
df_final.shape

(844993, 453)

In [48]:
df_final = df_final.sort_values(["datetime_hour"])

In [49]:
with pd.option_context('display.max_rows', None):
    print(df_final.isnull().sum())

datetime_hour                         0
region_id                             0
day_dew                               0
day_precip                            0
day_precipprob                        0
day_precipcover                       0
day_moonphase                         0
hour_temp                             0
hour_feelslike                        0
hour_humidity                         0
hour_dew                              0
hour_precip                           0
hour_precipprob                       0
hour_snow                             0
hour_snowdepth                        0
hour_windgust                         0
hour_windspeed                        0
hour_winddir                          0
hour_pressure                         0
hour_cloudcover                       0
hour_solarradiation                   0
hour_solarenergy                      0
hour_uvindex                          0
hour_conditions_simple_Clear          0
hour_conditions_simple_Cloudy         0


In [50]:
df_final = df_final.fillna(0)

In [51]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\409095542.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\409095542.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\409095542.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [52]:
df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
df_final['tg_intensity_zscore'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: (x - x.rolling(24, min_periods=1).mean()) / (x.rolling(24, min_periods=1).std() + 1e-9)))

C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\3857173777.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\3857173777.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_33600\3857173777.py:3: PerformanceWa

In [53]:
df_final.sample(20)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg

# ТУТ БУДЕ EDA

## Weather, ISW, Telegram shift for further models trainings

In [54]:
df_to_train = df_final.copy()
df_to_train = df_to_train.sort_values(['region_id', 'datetime_hour'])

In [55]:
isw_cols = [c for c in df_to_train.columns if 'isw_' in c]
df_to_train[isw_cols] = df_to_train.groupby('region_id')[isw_cols].shift(24).fillna(0)

In [56]:
df_to_train.head()

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg

In [57]:
tg_cols = [c for c in df_to_train.columns if 'tg_' in c]
df_to_train[tg_cols] = df_to_train.groupby('region_id')[tg_cols].shift(1).fillna(0)

In [58]:
df_to_train.head(5)

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg

In [59]:
hour_weather_cols = [c for c in df_to_train.columns if c.startswith('hour_')]
for col in hour_weather_cols:
    if df_to_train[col].dtype == bool:
        df_to_train[col] = df_to_train.groupby('region_id')[col].shift(1).fillna(False)
    else:
        df_to_train[col] = df_to_train.groupby('region_id')[col].shift(1).fillna(0)

In [63]:
day_weather_cols = [c for c in df_to_train.columns if (c.startswith('day_') and c not in ['day_datetime', 'day_sunrise', 'day_sunset', 'day_moonphase', 'day_of_week'])]
df_to_train[day_weather_cols] = df_to_train.groupby('region_id')[day_weather_cols].shift(24).fillna(0)

In [64]:
df_to_train.isna().sum()

datetime_hour          0
region_id              0
day_dew                0
day_precip             0
day_precipprob         0
                      ..
tg_topic_max           0
tg_topic_entropy       0
tg_velocity_3h         0
tg_intensity_ema_6h    0
tg_intensity_zscore    0
Length: 460, dtype: int64

In [65]:
df_to_train['alarm_minutes_in_hour'] = (df_to_train.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))

In [66]:
df_to_train.head()

,datetime_hour,region_id,day_dew,day_precip,day_precipprob,day_precipcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg

In [67]:
df_to_train.to_parquet("final_merged_dataset.parquet", index = False, engine = "pyarrow")